# 03 — SageMaker Feature Store (Feature Group + Ingestion)

This notebook:
- Reads curated CSV from S3 (produced in Notebook 01)
- Creates supervised lag features
- Creates a **SageMaker Feature Store Feature Group**
- Ingests the features (offline + optional online store)

This satisfies the course demo requirement: **feature store and feature groups**.


In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

import time
import boto3
import sagemaker
import pandas as pd
import s3fs

from src.feature_engineering import make_supervised
from src.feature_engineering import time_split  # optional
from sagemaker.feature_store.feature_group import FeatureGroup

In [ ]:
# Load from previous notebook
%store -r bucket
%store -r region
%store -r CURATED_PREFIX
%store -r BUOY_IDS

print("Bucket:", bucket)
print("Region:", region)
print("CURATED_PREFIX:", CURATED_PREFIX)
print("BUOY_IDS:", BUOY_IDS)

In [ ]:
# Read all curated CSV from S3
fs = s3fs.S3FileSystem()

csv_glob = f"s3://{bucket}/{CURATED_PREFIX}/buoy=*/stdmet.csv"
csv_paths = fs.glob(csv_glob.replace("s3://",""))  # s3fs glob expects bucket/key without scheme

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found at {csv_glob}")

dfs = []
for key in sorted(csv_paths):
    s3_uri = "s3://" + key
    df = pd.read_csv(s3_uri)
    dfs.append(df)

all_data = pd.concat(dfs, ignore_index=True)
all_data["timestamp"] = pd.to_datetime(all_data["timestamp"], errors="coerce", utc=True)
all_data = all_data.dropna(subset=["timestamp"]).sort_values(["station_id","timestamp"])

print("Rows:", len(all_data))
print("Buoys:", all_data["station_id"].nunique())
all_data.head()

In [ ]:
# Build supervised features (lags + target)
df_features, feature_cols = make_supervised(
    all_data,
    entity_col="station_id",
    time_col="timestamp",
    lead_hours=1,
    lags=(1, 2, 3, 6),
    include_current_features=False,
)

# Feature Store metadata
df_features["event_time"] = (
    pd.to_datetime(df_features["timestamp"], utc=True).astype("int64") / 1e9
).astype(float)

df_features["record_id"] = (
    df_features["station_id"].astype(str)
    + "_"
    + df_features["event_time"].astype(int).astype(str)
)

# Keep timestamp for debug, but you can drop it if you want
print("Supervised rows:", len(df_features))
df_features.head()

In [ ]:
# Create Feature Group (unique name per run)
sess = sagemaker.Session()
role = sagemaker.get_execution_role()

FEATURE_GROUP_NAME = f"buoy-wave-features-{int(time.time())}"
OFFLINE_S3_URI = f"s3://{bucket}/feature-store/offline/buoycast/"

ENABLE_ONLINE_STORE = True  # set False if you only want offline

feature_group = FeatureGroup(
    name=FEATURE_GROUP_NAME,
    sagemaker_session=sess,
)

feature_group.load_feature_definitions(data_frame=df_features[
    ["record_id","event_time","station_id"] + feature_cols + ["target_E_star"]
])

print("Creating feature group:", FEATURE_GROUP_NAME)
feature_group.create(
    record_identifier_name="record_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=ENABLE_ONLINE_STORE,
    s3_uri=OFFLINE_S3_URI,
)

feature_group.wait_for_create()
print("FeatureGroupStatus:", feature_group.describe()["FeatureGroupStatus"])

In [ ]:
# Ingest into Feature Store
ingest_df = df_features[
    ["record_id","event_time","station_id"] + feature_cols + ["target_E_star"]
].copy()

feature_group.ingest(
    data_frame=ingest_df,
    max_workers=4,
    wait=True,
)

desc = feature_group.describe()
print("FeatureGroupStatus:", desc["FeatureGroupStatus"])
print("OfflineStoreStatus:", desc.get("OfflineStoreStatus"))
print("OnlineStoreConfig:", desc.get("OnlineStoreConfig"))

In [ ]:
# Optional: verify online store retrieval (only works if ENABLE_ONLINE_STORE=True)
if ENABLE_ONLINE_STORE:
    fs_runtime = boto3.client("sagemaker-featurestore-runtime", region_name=region)

    sample_id = ingest_df["record_id"].iloc[0]
    rec = fs_runtime.get_record(FeatureGroupName=FEATURE_GROUP_NAME, RecordIdentifierValueAsString=sample_id)
    rec

In [ ]:
# Save for next notebooks
%store FEATURE_GROUP_NAME
%store OFFLINE_S3_URI